In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Створи плагін транспілятора

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';





<details>
<summary><b>Версії пакетів</b></summary>

Код на цій сторінці розроблено з використанням наступних залежностей.
Рекомендуємо використовувати ці або новіші версії.

```
qiskit[all]~=2.3.0
```
</details>
Створення [плагіна транспілятора](transpiler-plugins) — чудовий спосіб поділитися своїм кодом транспіляції з широкою спільнотою Qiskit, дозволяючи іншим користувачам скористатися функціональністю, яку ти розробив. Дякуємо за інтерес до розвитку спільноти Qiskit!

Перш ніж створювати плагін транспілятора, потрібно визначити, який тип плагіна підходить для твоєї ситуації. Існують три типи плагінів транспілятора:
- [**Плагін стадії транспілятора**](https://docs.quantum.ibm.com/api/qiskit/transpiler_plugins). Обери цей варіант, якщо ти визначаєш менеджер проходів, який можна підставити замість однієї з [6 стадій](transpiler-stages) заздалегідь налаштованого поетапного менеджера проходів.
- [**Плагін синтезу унітарного оператора**](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin). Обери цей варіант, якщо твій код транспіляції приймає на вхід унітарну матрицю (представлену як масив Numpy) і повертає опис квантового Circuit, що реалізує цей унітарний оператор.
- [**Плагін синтезу високорівневих об'єктів**](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin). Обери цей варіант, якщо твій код транспіляції приймає на вхід "високорівневий об'єкт" — наприклад, оператор Кліффорда або лінійну функцію — і повертає опис квантового Circuit, що реалізує цей об'єкт. Високорівневі об'єкти представлені підкласами класу [Operation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Operation).

Після того як ти визначив тип плагіна, виконай ці кроки для його створення:

1. Створи підклас відповідного абстрактного класу плагіна:
   - [PassManagerStagePlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.preset_passmanagers.plugin.PassManagerStagePlugin) для плагіна стадії транспілятора,
   - [UnitarySynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin) для плагіна синтезу унітарного оператора, і
   - [HighLevelSynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin) для плагіна синтезу високорівневих об'єктів.
2. Оголоси клас як [точку входу setuptools](https://setuptools.pypa.io/en/latest/userguide/entry_point.html) у метаданих пакета — зазвичай це робиться шляхом редагування файлу `pyproject.toml`, `setup.cfg` або `setup.py` твого Python-пакета.

Кількість плагінів в одному пакеті необмежена, але кожен плагін повинен мати унікальне ім'я. Сам Qiskit SDK включає кілька плагінів, чиї імена також зарезервовані. Зарезервовані імена:

- Плагіни стадій транспілятора: дивись [цю таблицю](https://docs.quantum.ibm.com/api/qiskit/transpiler_plugins#plugin-stages).
- Плагіни синтезу унітарного оператора: `default`, `aqc`, `sk`
- Плагіни синтезу високорівневих об'єктів:

| Клас операції | Назва операції | Зарезервовані імена |
| --- | --- | --- |
| [Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford#clifford) | `clifford` | `default`, `ag`, `bm`, `greedy`, `layers`, `lnn` |
| [LinearFunction](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction#linearfunction) | `linear_function` | `default`, `kms`, `pmh` |
| [PermutationGate](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.PermutationGate#permutationgate) | `permutation` | `default`, `kms`, `basic`, `acg`, `token_swapper` |

У наступних розділах ми наводимо приклади цих кроків для різних типів плагінів. У цих прикладах передбачається, що ми створюємо Python-пакет під назвою `my_qiskit_plugin`. Щоб дізнатися більше про створення Python-пакетів, можна ознайомитися з [цим посібником](https://packaging.python.org/en/latest/tutorials/packaging-projects/) на сайті Python.
## Приклад: Створення плагіна стадії транспілятора
У цьому прикладі ми створюємо плагін стадії транспілятора для стадії `layout` (опис 6 стадій вбудованого конвеєра транспіляції Qiskit дивись у розділі [Стадії транспілятора](transpiler-stages)).
Наш плагін просто запускає [VF2Layout](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.VF2Layout) з кількістю спроб, що залежить від запитаного рівня оптимізації.

Спочатку ми створюємо підклас [PassManagerStagePlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.preset_passmanagers.plugin.PassManagerStagePlugin). Потрібно реалізувати один метод — [`pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.preset_passmanagers.plugin.PassManagerStagePlugin#pass_manager). Цей метод приймає на вхід [PassManagerConfig](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManagerConfig) і повертає менеджер проходів, який ми визначаємо. Об'єкт PassManagerConfig зберігає інформацію про цільовий Backend, наприклад про карту зв'язності та базові Gate.

In [1]:
# This import is needed for python versions prior to 3.10
from __future__ import annotations

from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import VF2Layout
from qiskit.transpiler.passmanager_config import PassManagerConfig
from qiskit.transpiler.preset_passmanagers import common
from qiskit.transpiler.preset_passmanagers.plugin import (
    PassManagerStagePlugin,
)


class MyLayoutPlugin(PassManagerStagePlugin):
    def pass_manager(
        self,
        pass_manager_config: PassManagerConfig,
        optimization_level: int | None = None,
    ) -> PassManager:
        layout_pm = PassManager(
            [
                VF2Layout(
                    coupling_map=pass_manager_config.coupling_map,
                    properties=pass_manager_config.backend_properties,
                    max_trials=optimization_level * 10 + 1,
                    target=pass_manager_config.target,
                )
            ]
        )
        layout_pm += common.generate_embed_passmanager(
            pass_manager_config.coupling_map
        )
        return layout_pm

Тепер ми оголошуємо плагін, додавши точку входу до метаданих нашого Python-пакета.
Тут передбачається, що визначений нами клас доступний у модулі `my_qiskit_plugin` — наприклад, імпортований у файлі `__init__.py` модуля `my_qiskit_plugin`.
Редагуємо файл `pyproject.toml`, `setup.cfg` або `setup.py` нашого пакета (залежно від того, який файл ти обрав для зберігання метаданих Python-проєкту):

<Tabs>
  <TabItem value="package-table-toml" label="pyproject.toml" default>
    ```toml
    [project.entry-points."qiskit.transpiler.layout"]
    "my_layout" = "my_qiskit_plugin:MyLayoutPlugin"
    ```
  </TabItem>
  <TabItem value="package-table-cfg" label="setup.cfg">
    ```ini
    [options.entry_points]
    qiskit.transpiler.layout =
        my_layout = my_qiskit_plugin:MyLayoutPlugin
    ```
  </TabItem>
  <TabItem value="package-table-py" label="setup.py">
    ```python
    from setuptools import setup

    setup(
        # ...,
        entry_points={
            'qiskit.transpiler.layout': [
                'my_layout = my_qiskit_plugin:MyLayoutPlugin',
            ]
        }
    )
    ```
  </TabItem>
</Tabs>
Дивись [таблицю стадій плагінів транспілятора](https://docs.quantum.ibm.com/api/qiskit/transpiler_plugins#stage-table), де вказані точки входу та вимоги для кожної стадії транспілятора.

Щоб переконатися, що Qiskit успішно виявляє твій плагін, встанови пакет з плагіном і дотримуйся інструкцій у розділі [Плагіни транспілятора](transpiler-plugins#list-available-transpiler-stage-plugins) для перегляду списку встановлених плагінів — переконайся, що твій плагін є в списку:

In [2]:
from qiskit.transpiler.preset_passmanagers.plugin import list_stage_plugins

list_stage_plugins("layout")

['default', 'dense', 'sabre', 'trivial']

If our example plugin were installed, then the name `my_layout` would appear in this list.

If you want to use a built-in transpiler stage as the starting point for your transpiler stage plugin, you can obtain the pass manager for a built-in transpiler stage using [PassManagerStagePluginManager](/docs/api/qiskit/qiskit.transpiler.preset_passmanagers.plugin.PassManagerStagePluginManager#passmanagerstagepluginmanager). The following code cell shows how to do this to obtain the built-in optimization stage for optimization level 3.

In [3]:
from qiskit.transpiler.preset_passmanagers.plugin import (
    PassManagerStagePluginManager,
)

# Initialize the plugin manager
plugin_manager = PassManagerStagePluginManager()

# Here we create a pass manager config to use as an example.
# Instead, you should use the pass manager config that you already received as input
# to the pass_manager method of your PassManagerStagePlugin.
pass_manager_config = PassManagerConfig()

# Obtain the desired built-in transpiler stage
optimization = plugin_manager.get_passmanager_stage(
    "optimization", "default", pass_manager_config, optimization_level=3
)

Якби наш приклад плагіна було встановлено, то назва `my_layout` з'явилася б у цьому списку.

Якщо ти хочеш взяти вбудовану стадію транспілятора за основу для свого плагіна, можна отримати менеджер проходів вбудованої стадії за допомогою [PassManagerStagePluginManager](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.preset_passmanagers.plugin.PassManagerStagePluginManager#passmanagerstagepluginmanager). Наступний блок коду демонструє, як отримати вбудовану стадію оптимізації для рівня оптимізації 3.

In [4]:
import numpy as np
from qiskit.circuit import QuantumCircuit, QuantumRegister
from qiskit.converters import circuit_to_dag
from qiskit.dagcircuit.dagcircuit import DAGCircuit
from qiskit.quantum_info import Operator
from qiskit.transpiler.passes import UnitarySynthesis
from qiskit.transpiler.passes.synthesis.plugin import UnitarySynthesisPlugin


class MyUnitarySynthesisPlugin(UnitarySynthesisPlugin):
    @property
    def supports_basis_gates(self):
        # Returns True if the plugin can target a list of basis gates
        return True

    @property
    def supports_coupling_map(self):
        # Returns True if the plugin can synthesize for a given coupling map
        return False

    @property
    def supports_natural_direction(self):
        # Returns True if the plugin supports a toggle for considering
        # directionality of 2-qubit gates
        return False

    @property
    def supports_pulse_optimize(self):
        # Returns True if the plugin can optimize pulses during synthesis
        return False

    @property
    def supports_gate_lengths(self):
        # Returns True if the plugin can accept information about gate lengths
        return False

    @property
    def supports_gate_errors(self):
        # Returns True if the plugin can accept information about gate errors
        return False

    @property
    def supports_gate_lengths_by_qubit(self):
        # Returns True if the plugin can accept information about gate lengths
        # (The format of the input differs from supports_gate_lengths)
        return False

    @property
    def supports_gate_errors_by_qubit(self):
        # Returns True if the plugin can accept information about gate errors
        # (The format of the input differs from supports_gate_errors)
        return False

    @property
    def min_qubits(self):
        # Returns the minimum number of qubits the plugin supports
        return None

    @property
    def max_qubits(self):
        # Returns the maximum number of qubits the plugin supports
        return None

    @property
    def supported_bases(self):
        # Returns a dictionary of supported bases for synthesis
        return None

    def run(self, unitary: np.ndarray, **options) -> DAGCircuit:
        basis_gates = options["basis_gates"]
        synth_pass = UnitarySynthesis(basis_gates, min_qubits=3)
        qubits = QuantumRegister(3)
        circuit = QuantumCircuit(qubits)
        circuit.append(Operator(unitary).to_instruction(), qubits)
        dag_circuit = synth_pass.run(circuit_to_dag(circuit))
        return dag_circuit

## Приклад: Створення плагіна синтезу унітарного оператора
У цьому прикладі ми створимо плагін синтезу унітарного оператора, який просто використовує вбудований прохід транспіляції [UnitarySynthesis](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.UnitarySynthesis#unitarysynthesis) для синтезу Gate. Звісно, твій власний плагін робитиме щось цікавіше.

Клас [UnitarySynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin) визначає інтерфейс і контракт для плагінів синтезу унітарного оператора. Основний метод —
[`run`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin#run),
який приймає на вхід масив Numpy, що зберігає унітарну матрицю,
і повертає [DAGCircuit](https://docs.quantum.ibm.com/api/qiskit/qiskit.dagcircuit.DAGCircuit), що представляє Circuit, синтезований з цієї унітарної матриці.
Крім методу `run`, необхідно визначити низку методів-властивостей.
Документацію всіх обов'язкових властивостей дивись у [UnitarySynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin).

Давай створимо наш підклас UnitarySynthesisPlugin:

In [5]:
from qiskit.transpiler.passes.synthesis import unitary_synthesis_plugin_names

unitary_synthesis_plugin_names()

['aqc', 'clifford', 'default', 'gridsynth', 'sk']

Якщо тобі не вистачає вхідних даних, доступних у методі [`run`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin#run),
для твоїх цілей — будь ласка, [відкрий issue](https://github.com/Qiskit/qiskit/issues/new/choose) із поясненням своїх вимог. Зміни в інтерфейсі плагіна, наприклад додавання нових необов'язкових вхідних параметрів, виконуватимуться зі збереженням зворотної сумісності, щоб не вимагати змін від існуючих плагінів.

> **Note:** Усі методи з префіксом `supports_` зарезервовані в похідному класі `UnitarySynthesisPlugin` як частина інтерфейсу. Не слід визначати власні методи `supports_*` у підкласі, якщо вони не визначені в абстрактному класі.

Тепер ми оголошуємо плагін, додавши точку входу до метаданих нашого Python-пакета.
Тут передбачається, що визначений нами клас доступний у модулі `my_qiskit_plugin` — наприклад, імпортований у файлі `__init__.py` модуля `my_qiskit_plugin`.
Редагуємо файл `pyproject.toml`, `setup.cfg` або `setup.py` нашого пакета:

<Tabs>
  <TabItem value="package-table-toml" label="pyproject.toml" default>
    ```toml
    [project.entry-points."qiskit.unitary_synthesis"]
    "my_unitary_synthesis" = "my_qiskit_plugin:MyUnitarySynthesisPlugin"
    ```
  </TabItem>
  <TabItem value="package-table-cfg" label="setup.cfg">
    ```ini
    [options.entry_points]
    qiskit.unitary_synthesis =
        my_unitary_synthesis = my_qiskit_plugin:MyUnitarySynthesisPlugin
    ```
  </TabItem>
  <TabItem value="package-table-py" label="setup.py">
    ```python
    from setuptools import setup

    setup(
        # ...,
        entry_points={
            'qiskit.unitary_synthesis': [
                'my_unitary_synthesis = my_qiskit_plugin:MyUnitarySynthesisPlugin',
            ]
        }
    )
    ```
  </TabItem>
</Tabs>

Як і раніше, якщо твій проєкт використовує `setup.cfg` або `setup.py` замість `pyproject.toml`, дивись [документацію setuptools](https://setuptools.pypa.io/en/latest/userguide/entry_point.html), щоб дізнатися, як адаптувати ці рядки для твоєї ситуації.

Щоб переконатися, що Qiskit успішно виявляє твій плагін, встанови пакет з плагіном і дотримуйся інструкцій у розділі [Плагіни транспілятора](transpiler-plugins#list-available-unitary-synthesis-plugins) для перегляду списку встановлених плагінів — переконайся, що твій плагін є в списку:

In [6]:
from qiskit.synthesis import synth_clifford_bm
from qiskit.transpiler.passes.synthesis.plugin import HighLevelSynthesisPlugin


class MyCliffordSynthesisPlugin(HighLevelSynthesisPlugin):
    def run(
        self,
        high_level_object,
        coupling_map=None,
        target=None,
        qubits=None,
        **options,
    ) -> QuantumCircuit:
        if high_level_object.num_qubits <= 3:
            return synth_clifford_bm(high_level_object)
        else:
            return None

This plugin synthesizes objects of type [Clifford](/docs/api/qiskit/qiskit.quantum_info.Clifford) that have
at most 3 qubits, using the `synth_clifford_bm` method.

Now, we expose the plugin by adding an entry point in our Python package metadata.
Here, we assume that the class we defined is exposed in a module called `my_qiskit_plugin`, for example by being imported in the `__init__.py` file of the `my_qiskit_plugin` module.
We edit the `pyproject.toml`, `setup.cfg`, or `setup.py` file of our package:

<Tabs>
  <TabItem value="package-table-toml" label="pyproject.toml" default>
    ```toml
    [project.entry-points."qiskit.synthesis"]
    "clifford.my_clifford_synthesis" = "my_qiskit_plugin:MyCliffordSynthesisPlugin"
    ```
  </TabItem>
  <TabItem value="package-table-cfg" label="setup.cfg">
    ```ini
    [options.entry_points]
    qiskit.synthesis =
        clifford.my_clifford_synthesis = my_qiskit_plugin:MyCliffordSynthesisPlugin
    ```
  </TabItem>
  <TabItem value="package-table-py" label="setup.py">
    ```python
    from setuptools import setup

    setup(
        # ...,
        entry_points={
            'qiskit.synthesis': [
                'clifford.my_clifford_synthesis = my_qiskit_plugin:MyCliffordSynthesisPlugin',
            ]
        }
    )
    ```
  </TabItem>
</Tabs>

The `name` consists of two parts separated by a dot (`.`):
- The name of the type of [Operation](/docs/api/qiskit/qiskit.circuit.Operation) that the plugin synthesizes (in this case, `clifford`). Note that this string corresponds to the [`name`](/docs/api/qiskit/qiskit.circuit.Operation#name) attribute of the Operation class, and not the name of the class itself.
- The name of the plugin (in this case, `special`).

As before, if your project uses `setup.cfg` or `setup.py` instead of `pyproject.toml`, see the [setuptools documentation](https://setuptools.pypa.io/en/latest/userguide/entry_point.html) for how to adapt these lines for your situation.

To check that your plugin is successfully detected by Qiskit, install your plugin package and follow the instructions at [Transpiler plugins](transpiler-plugins#list-available-high-level-synthesis-plugins) for listing installed plugins, and ensure that your plugin appears in the list:

In [7]:
from qiskit.transpiler.passes.synthesis import (
    high_level_synthesis_plugin_names,
)

high_level_synthesis_plugin_names("clifford")

['ag', 'bm', 'default', 'greedy', 'layers', 'lnn', 'rb_default']

Якби наш приклад плагіна було встановлено, то назва `my_unitary_synthesis` з'явилася б у цьому списку.

Щоб підтримати плагіни синтезу унітарного оператора, які надають кілька опцій,
інтерфейс плагіна передбачає можливість передачі користувачем словника конфігурації у вільній формі. Він передаватиметься до методу `run`
через іменований аргумент `options`. Якщо твій плагін має такі параметри конфігурації — обов'язково задокументуй їх.

## Приклад: Створення плагіна синтезу високорівневих об'єктів

У цьому прикладі ми створимо плагін синтезу високорівневих об'єктів, який просто використовує вбудовану функцію [synth_clifford_bm](https://docs.quantum.ibm.com/api/qiskit/synthesis#synth_clifford_bm) для синтезу оператора Кліффорда.

Клас [HighLevelSynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin) визначає інтерфейс і контракт для плагінів синтезу високорівневих об'єктів. Основний метод — [`run`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin#run).
Позиційний аргумент `high_level_object` — це [Operation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Operation), що представляє "високорівневий" об'єкт для синтезу. Наприклад, це може бути
[LinearFunction](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction) або
[Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford).
Присутні такі іменовані аргументи:
- `target` вказує цільовий Backend, надаючи плагіну доступ до всієї специфічної для цілі інформації,
наприклад карти зв'язності, підтримуваного набору Gate тощо.
- `coupling_map` вказує лише карту зв'язності і використовується тільки тоді, коли `target` не задано.
- `qubits` вказує список Qubit, на яких визначено
високорівневий об'єкт — у разі, якщо синтез виконується на фізичному Circuit.
Значення ``None`` означає, що розміщення ще не вибрано і фізичні Qubit в цілі або карті зв'язності, на яких виконується ця операція, ще не визначено.
- `options` — словник конфігурації у вільній формі для специфічних опцій плагіна. Якщо твій плагін має такі параметри конфігурації —
обов'язково задокументуй їх.

Метод `run` повертає [QuantumCircuit](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit),
що представляє Circuit, синтезований з цього високорівневого об'єкта.
Також допустимо повертати `None`, що означає нездатність плагіна синтезувати даний високорівневий об'єкт.
Безпосередньо синтез високорівневих об'єктів виконується проходом транспілятора
[HighLevelSynthesis](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.HighLevelSynthesis).

Крім методу `run`, необхідно визначити низку методів-властивостей.
Документацію всіх обов'язкових властивостей дивись у [HighLevelSynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin).

Давай визначимо наш підклас HighLevelSynthesisPlugin: